# TAPAS privacy benchmark: CTGAN (non-DP neural) — Colab T4 runner

Runs `benchmark_tapas/scripts/run_ctgan.py` (5-attack MIA battery) on a GPU runtime.

**Before running, upload to your Drive at `MyDrive/VRI/experimentation/` (mirroring the repo layout):**
- `benchmark_tapas/config.py`, `benchmark_tapas/common.py`, `benchmark_tapas/scripts/run_ctgan.py`
- `data/adult_train.csv`, `data/adult_test.csv`

**Robustness (built in):** Drive-mounted so nothing is lost on disconnect; each attack's result is checkpointed to Drive *as it finishes*; re-running **resumes** (finished attacks + memoised generator fits are skipped). Just re-run the whole notebook after a cut-off session.

> Set the neural `n_iter` in `config.py` (CTGAN_N_ITER / DPGAN_N_ITER) from `convergence_check.py` before the real run — the default 200 is provisional.

## 1. Install pinned deps (restarts runtime — expected)

In [ ]:
# TAPAS's pyproject.toml declares `python = ">=3.9, <3.11"` (poetry-core
# build backend) -- Colab runs Python 3.12+, which poetry-core rejects
# outright when computing build metadata ("Getting requirements to build
# wheel did not run successfully"). Clone the exact pinned commit and patch
# that constraint before installing, rather than installing directly from
# git (which can't be patched first).
!rm -rf /content/tapas_src
!git clone -q https://github.com/alan-turing-institute/tapas.git /content/tapas_src
!cd /content/tapas_src && git checkout -q a7069d7e040828db0da174d1b003fa03a98e5453
!sed -i 's/python = ">=3.9, <3.11"/python = ">=3.9"/' /content/tapas_src/pyproject.toml
!grep '^python =' /content/tapas_src/pyproject.toml  # sanity-check the patch applied

# TAPAS pins pandas ^1.4.1 (wants <2.0); old pandas 1.x has no py3.12 wheel and
# fails to build from source. We know from the local runs that TAPAS's code
# works fine with pandas 2.x -- so install TAPAS with --no-deps and pin the rest
# explicitly (same approach as the eff_eps colab notebook).
!pip install /content/tapas_src --no-deps -q
!pip install palettable==3.3.3 -q
!pip install synthcity==0.2.12 -q
!pip install opacus==1.4.1 -q
!pip install pandas==2.3.3 -q

import os
os.kill(os.getpid(), 9)  # force restart to clear numpy/torch binary conflicts

In [ ]:
import tapas, synthcity
from synthcity.plugins import Plugins
print('imports OK: tapas + synthcity')

## 2. Verify GPU (stop here if CUDA is False)

In [ ]:
import torch
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('\n*** No CUDA GPU. Runtime > Change runtime type > T4 GPU, then re-run. ***')
    print("Synthcity's ctgan/dpgan plugins default to device='cpu' and do NOT")
    print('auto-detect CUDA -- the run_*.py script detects it here and passes')
    print('device="cuda" through, but only if this prints True.')
else:
    print('GPU OK -- run script will train on CUDA.')
!nvidia-smi

## 3. Mount Drive + symlink benchmark_tapas/ (all writes go to Drive)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Everything writes straight to Drive so nothing is lost on a runtime reset.
DRIVE_BASE = '/content/drive/MyDrive/VRI/experimentation'
os.chdir('/content')

# Symlink the WHOLE benchmark_tapas/ folder from Drive into /content. Because
# config.py / common.py resolve their own paths via Path(__file__).resolve()
# (which follows this symlink through to the real Drive path), the cache/,
# results/, log files -- and, via REPO_ROOT, the data/ folder -- all resolve
# straight onto Drive. So: (1) checkpoints go to Drive; (2) run_attack writes
# each attack's result JSON to Drive AS IT FINISHES (incremental checkpoint);
# (3) re-running skips any attack whose JSON already exists on Drive -- a
# disconnected session RESUMES rather than restarts.
os.makedirs('/content/benchmark_tapas', exist_ok=True) if False else None
if not os.path.exists('/content/benchmark_tapas'):
    os.symlink(f'{DRIVE_BASE}/benchmark_tapas', '/content/benchmark_tapas')

required = [
    f'{DRIVE_BASE}/benchmark_tapas/config.py',
    f'{DRIVE_BASE}/benchmark_tapas/common.py',
    f'{DRIVE_BASE}/benchmark_tapas/scripts/run_ctgan.py',
    f'{DRIVE_BASE}/data/adult_train.csv',
    f'{DRIVE_BASE}/data/adult_test.csv',
]
missing = [p for p in required if not os.path.exists(p)]
if missing:
    print('*** MISSING on Drive -- upload these before running: ***')
    for p in missing: print('  ', p)
else:
    print('All required files present on Drive. Symlink ready.')
    print('cache so far:', os.listdir(f'{DRIVE_BASE}/benchmark_tapas/cache/ctgan')
          if os.path.exists(f'{DRIVE_BASE}/benchmark_tapas/cache/ctgan') else '(none yet)')

## 3c. ONE-TIME recovery cleanup (pickle-bug)

Deletes the **corrupt** `ctgan`/`dpgan` cache + partial results on Drive (from the pre-fix crash) so the fixed run starts clean. **Guarded**: set `RUN_CLEANUP = True`, run this cell once, then set it back to `False` — so a later *Run all* never wipes good results. Only needed once, after uploading the fixed `common.py`.

In [ ]:
# ONE-TIME cleanup after the pickle-bug crash. Guarded so 'Run all' can't nuke
# good results: flip RUN_CLEANUP to True, run once, then set it back to False.
RUN_CLEANUP = False
if RUN_CLEANUP:
    import shutil, os
    B = '/content/drive/MyDrive/VRI/experimentation/benchmark_tapas'
    for m in ['ctgan', 'dpgan']:
        for sub in ['cache', 'results']:
            p = f'{B}/{sub}/{m}'
            if os.path.exists(p):
                shutil.rmtree(p); print('deleted', p)
    print('cleanup done -- set RUN_CLEANUP back to False now')
else:
    print('cleanup skipped -- set RUN_CLEANUP=True to delete corrupt ctgan/dpgan cache+results, then re-run this cell')

## 3b. Pre-flight timing probe (run BEFORE the audit)

Times 3 real 500-row fits on the T4 and projects the first attack (Groundhog) time + whether the current `num_train`/`num_test` fit one ~2 hr session. This is a **time** check, not a statistical one.

- **FITS** for both methods -> go straight to the audit below. If it reports headroom, you *may* raise the neural counts in `config.py` (re-upload it) for tighter CIs.
- **OVER** -> use the smaller counts it suggests (edit `config.py`, re-upload) before running the audit, so the first attack can actually complete.

Needs Drive mounted + symlinked (cell above) first. The whole ~60-fit generator cost is inside Groundhog and isn't checkpointed until it finishes, so a Groundhog that can't fit one session never lands.

In [ ]:
!python /content/benchmark_tapas/neural_tuning/probe_fit_time.py ctgan dpgan

## 4. Run the audit (safe to re-run after a disconnect — it resumes)

In [ ]:
!python /content/benchmark_tapas/scripts/run_ctgan.py

In [ ]:
!python /content/benchmark_tapas/scripts/run_dpgan.py

## 5. Read back results / show resumable checkpoints

In [ ]:
import pandas as pd, os, glob
DRIVE_BASE = '/content/drive/MyDrive/VRI/experimentation'
res = f'{DRIVE_BASE}/benchmark_tapas/results/per_method/ctgan/effeps_ctgan.csv'
if os.path.exists(res):
    print(pd.read_csv(res)[['attack','auc','eps_low_95','eps_high_95','wall_time_s']].to_string(index=False))
else:
    print('No effeps CSV yet. Completed attack caches on Drive:')
    for p in sorted(glob.glob(f'{DRIVE_BASE}/benchmark_tapas/cache/ctgan/result_*.json')):
        print('  ', os.path.basename(p))

res = f'{DRIVE_BASE}/benchmark_tapas/results/per_method/dpgan/effeps_dpgan.csv'
if os.path.exists(res):
    print(pd.read_csv(res)[['attack','auc','eps_low_95','eps_high_95','wall_time_s']].to_string(index=False))
else:
    print('No effeps CSV yet. Completed attack caches on Drive:')
    for p in sorted(glob.glob(f'{DRIVE_BASE}/benchmark_tapas/cache/dpgan/result_*.json')):
        print('  ', os.path.basename(p))